# Bauwerk Decision-Transformer Experiments
In this notebook, we'll lay out initial experiments to validate the Decision Transformer architecture in a controlled setting. We'll run three experiments:
1. **Overfit DT**: DT trained and evaluated on one task. 
2. **Underfit DT**: DT trained on one task and evaluated on others.
3. **General DT**: DT trained and evaluated on all tasks.

We compose the training sets of optimal state-action trajectories from the relevant task. In the future, we will relax this constraint and use trajectories collected by ~optimal specialist RL agents trained in the relevant task. Before we proceed to more complicated building simulations, DT must perform competently on tests 1 and 3 (task 2 will be used to understand how well DT can generalise across tasks, and we do not expect strong performance). As with previous experiments, we'll measure performance $p^{'}$ of our controller $p_{dt}$ relative to a random controller $p_{r}$ and an optimal controller $p_{o}$ via the following relationship

$$
p^{'} = \frac{p_{dt} - p_{r}}{p_{o} - p_{r}}
$$

First we'll import the required packages, then set up experiment 1.

In [1]:
import gym
import bauwerk
import bauwerk.benchmarks

import torch
import numpy as np
from tqdm import tqdm

from utils.utils import Cfg
from utils.utils import ObsWrapper
from agent.DT.agent import Agent
from data.tokenizer import Tokenizer

# load config
Cfg = Cfg()
cfg = Cfg.parse(model='dt')

# import tokenizer
tokenizer = Tokenizer(cfg.tokenizer)

model = Agent(cfg)

/Users/scottjeen/miniforge3/envs/odes/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We'll set up a helper function to calculate the performance of a rollout of actions of abitrary length, this will be useful for evaluating optimal and random actions.

In [27]:
def evaluate_actions(actions, env, prompt_steps):
    cum_reward = 0
    obs = env.reset()
    for i, action in enumerate(actions):
        obs, reward, done, info = env.step(np.array(action, dtype=np.float32))
        
        if i >= prompt_steps:
            cum_reward += reward

    return cum_reward / len(actions)

We need a function that creates a `prompt` for the agent. The `prompt` is a sequence of tokens to use as the agent's first input; in the Gato paper, the authors suggest the prompt primes the network for the task it is about to perform. In this case, we'll build the `prompt` using the first few optimal actions in the environment. We take as many actions as are required to create a prompt with length equal to the sequences used to pre-train the agent -- in our case that's 64 tokens.

In [2]:
def prompt(env, obs_dim, act_dim, prompt_steps):
    """
    Creates task-specifc tokenized prompt for DT.
    :param env: Bauwerk environment set to relevant task
    :param cfg: (dict) decision transformer config.
    :return tokenized prompt: array of state-action tokens, shape [context_length,]
    """
    print('...creating prompt...')
    optimal_actions = bauwerk.solve(env)[0]
    state_actions = []
    
    # create masks
    obs_mask = np.zeros(shape=(prompt_steps + 1, obs_dim + act_dim))  # +1 because we include final additional obs
    act_mask = np.zeros(shape=(prompt_steps, obs_dim + act_dim))
    obs_mask[:, :obs_dim] = np.arange(start=1, stop=obs_dim+1)
    act_mask[:, obs_dim: obs_dim + act_dim] = 1
    obs_mask = obs_mask.flatten()[-cfg.transformer.context_length:]
    act_mask = act_mask.flatten()[-cfg.transformer.context_length:]
    
    obs = env.reset()
    for step in range(prompt_steps):
        state_actions.append(obs)
        action = optimal_actions[step]
        obs, _, _, _ = env.step(action)
        state_actions.append(action)
    
    state_actions.append(obs)
    
    # correct masks for last obs
    obs_mask[:-obs_dim] = obs_mask[obs_dim:]
    obs_mask[-obs_dim:] = np.arange(start=1, stop=obs_dim+1)
    act_mask[:-obs_dim] = act_mask[obs_dim:]
    act_mask[-obs_dim:] = 0
    
    prompt = np.concatenate(np.array(state_actions, dtype=object))[-cfg.transformer.context_length:]  # flattened array sliced to context length 
    tokenised_prompt = tokenizer.tokenize(prompt) 
        
    return tokenised_prompt, obs_mask, act_mask

Now we'll create a function that sets up a task and performs an evalutive rollout with our agent. We need this function to step the simulator until the end of the prompt sequence, after which the agent can take control.

In [23]:
def rollout_with_prompt(model, env, eval_steps):
    act_dim = int(env.action_space.shape[0])
    obs_dim = 5
    model_actions = []
    optimal_actions = bauwerk.solve(env)[0]
    rollout_reward = 0
    prompt_steps = int(np.ceil(cfg.transformer.context_length / (obs_dim + act_dim)))  # +1 for 1-d action

    ### we have to create two ugly loops to give DT the env after prompt steps ###
    
    # loop 1: setting up env
    print('...preparing env...')
    obs = env.reset()
    for i in range(prompt_steps):
        action = optimal_actions[i]
        obs, _, _, _ = env.step(action)

    # create prompt sequence
    tokens, obs_mask, act_mask = prompt(env, obs_dim, act_dim, prompt_steps)
    
    # loop 2: eval rollout
    print('...collecting rollout...')
    for _ in tqdm(range(eval_steps)):
        action_dims = []
        # we predict action dimensions one-by-one
        for _ in range(act_dim):
            
            out_seq = model.predict_sequence(
                input_sequence=np.expand_dims(tokens, axis=0),
                obs_mask=np.expand_dims(obs_mask, axis=0),
                act_mask=np.expand_dims(act_mask, axis=0)
            )
            action_dims.append(out_seq[:, -1].detach().numpy())  # action dim is final dim of predicted sequence
            tokens, obs_mask, act_mask = tokenizer.update_sequences(tokens, obs_mask, act_mask, out_seq[:, -1], action=True)
            
        
        action = np.array(action_dims, dtype=np.float32).flatten()
        obs, reward, _, _ = env.step(action)
        
        obs_tokens = tokenizer.tokenize(obs)
        tokens, obs_mask, act_mask = tokenizer.update_sequences(tokens, obs_mask, act_mask, obs_tokens, obs=True)
        
        model_actions.append(action)
        rollout_reward += reward
        
    mean_reward = rollout_reward / len(model_actions)

    return mean_reward, model_actions

## Experiment 1: Overfit DT
Here, we'll import DT parameters fit to Bauwerk's $0^{th}$ Task. We'll evaluate DT on the same Task.


In [35]:
# import pre-trained model
model.load_state_dict(torch.load(cfg.model_path))
model.eval()

# setup env                   
build_dist_b = bauwerk.benchmarks.BuildDistB(seed=0)
env = build_dist_b.make_env()
env = ObsWrapper(env)
env.set_task(build_dist_b.train_tasks[0])
obs_dim = 5
act_dim = 1

# DT rollout
prompt_steps = int(np.ceil(cfg.transformer.context_length / (obs_dim + act_dim)))  # +1 for 1-d action
dt_reward, model_actions = rollout_with_prompt(model=model, env=env, eval_steps=8759 - prompt_steps)

# optimal rollout
optimal_actions = bauwerk.solve(env)
optimal_reward = evaluate_optimal(optimal_actions[0], env, 11)

# random rollout
random_actions = [env.action_space.sample() for _ in range(8759 - prompt_steps)]
random_reward = evaluate_optimal(random_actions, env, 11)

...preparing env...
obs before prompt [0.704 0.    0.    1.    0.   ]
...creating prompt...


/Users/scottjeen/phd/research/elizabeth-homes/exp/enjeeneer/rl-exp2/data/tokenizer.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32)


...collecting rollout...
obs after prompt [ 1.201  2.816  5.98  -0.966  0.259]


100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 8748/8748 [01:24<00:00, 103.00it/s]


In [36]:
p = (mean_reward - random_reward) / (optimal_reward - random_reward)
print('DT Overfit Performance: {:.2f}'.format(p))

DT Overfit Performance: 0.66
